In [0]:
%pip install -U google-genai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
api_key = dbutils.secrets.get(
    scope="gemini",
    key="GEMINI_API_KEY"
)

print("Gemini API key loaded successfully.")

Gemini API key loaded successfully.


In [0]:
%pip install -U typing_extensions

from google import genai

api_key = dbutils.secrets.get(
    scope="gemini",
    key="GEMINI_API_KEY"
)

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Gemini client initialized successfully.


In [0]:
%restart_python

In [0]:
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [0]:
import json

kpi_df = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_kpi_summary
""")

kpi_data = kpi_df.toPandas().to_dict(orient="records")

print(json.dumps(kpi_data, indent=2, default=str))

[
  {
    "start_date": "2022-03-31",
    "end_date": "2022-06-29",
    "total_orders": 120378,
    "cancelled_orders": 17185,
    "total_units_sold": 110992,
    "gross_order_value": "78592678.30",
    "net_revenue": "71673394.00",
    "average_order_value": "694.56",
    "cancellation_rate": "14.28"
  }
]


In [0]:
kpi_json = json.dumps(kpi_data, indent=2, default=str)

prompt = f"""
You are an e-commerce business analyst.

You must analyze ONLY the Databricks evidence provided below.
Do not invent numbers or facts.

DATAMART KPI EVIDENCE:
{kpi_json}

Prepare a concise business performance summary.

Structure your response as:

1. Executive Summary
2. Key Metrics
3. Important Observations
4. Areas That Need Investigation

Rules:
- Use the exact numbers from the evidence.
- Clearly distinguish observations from possible explanations.
- Do not claim causation unless the evidence establishes it.
- Do not invent recommendations unsupported by the data.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

### 1. Executive Summary

During the period from March 31, 2022, to June 29, 2022, the business generated a Gross Order Value of 78,592,678.30 and a Net Revenue of 71,673,394.00. Out of 120,378 total orders, 17,185 were cancelled, resulting in a cancellation rate of 14.28%. A total of 110,992 units were sold, with an Average Order Value of 694.56.

---

### 2. Key Metrics

* **Reporting Period:** 2022-03-31 to 2022-06-29
* **Total Orders:** 120,378
* **Cancelled Orders:** 17,185
* **Cancellation Rate:** 14.28%
* **Total Units Sold:** 110,992
* **Gross Order Value:** 78,592,678.30
* **Net Revenue:** 71,673,394.00
* **Average Order Value:** 694.56

---

### 3. Important Observations

* **Order Cancellations:** 17,185 orders out of 120,378 were cancelled, which represents 14.28% of all orders placed during this period.
* **Revenue Variance:** Net revenue (71,673,394.00) is 6,919,284.30 lower than Gross Order Value (78,592,678.30).
* **Order Volume vs. Units Sold:** Total units sold (110,9

In [0]:
spark.sql("""
SELECT *
FROM ecommerce_ai.ai.v_kpi_summary
""").show(truncate=False)

spark.sql("""
SELECT
    SUM(net_revenue) AS net_revenue,
    SUM(total_orders - cancelled_orders) AS non_cancelled_orders,
    SUM(net_revenue) /
        NULLIF(SUM(total_orders - cancelled_orders), 0) AS calculated_aov
FROM ecommerce_ai.gold.daily_sales
""").show()

+----------+----------+------------+----------------+----------------+-----------------+-----------+-------------------+-----------------+
|start_date|end_date  |total_orders|cancelled_orders|total_units_sold|gross_order_value|net_revenue|average_order_value|cancellation_rate|
+----------+----------+------------+----------------+----------------+-----------------+-----------+-------------------+-----------------+
|2022-03-31|2022-06-29|120378      |17185           |110992          |78592678.30      |71673394.00|694.56             |14.28            |
+----------+----------+------------+----------------+----------------+-----------------+-----------+-------------------+-----------------+

+-----------+--------------------+--------------+
|net_revenue|non_cancelled_orders|calculated_aov|
+-----------+--------------------+--------------+
|71673394.00|              103193|  694.55674319|
+-----------+--------------------+--------------+



In [0]:
import json

kpi = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_kpi_summary
""").toPandas().to_dict(orient="records")

categories = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_top_categories
    LIMIT 10
""").toPandas().to_dict(orient="records")

skus = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_top_skus
    LIMIT 10
""").toPandas().to_dict(orient="records")

inventory = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_inventory_risk
    LIMIT 15
""").toPandas().to_dict(orient="records")

fulfillment = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.v_fulfillment
""").toPandas().to_dict(orient="records")

anomalies = spark.sql("""
    SELECT *
    FROM ecommerce_ai.ai.anomaly_events
    ORDER BY Date DESC
""").toPandas().to_dict(orient="records")

print("Evidence loaded successfully.")
print("KPI:", len(kpi))
print("Categories:", len(categories))
print("SKUs:", len(skus))
print("Inventory risks:", len(inventory))
print("Fulfillment:", len(fulfillment))
print("Anomalies:", len(anomalies))

Evidence loaded successfully.
KPI: 1
Categories: 9
SKUs: 10
Inventory risks: 15
Fulfillment: 2
Anomalies: 20


In [0]:
evidence = {
    "kpi_summary": kpi,
    "top_categories": categories,
    "top_skus": skus,
    "inventory_risk": inventory,
    "fulfillment": fulfillment,
    "anomalies": anomalies
}

evidence_json = json.dumps(
    evidence,
    indent=2,
    default=str
)

print(evidence_json[:10000])

{
  "kpi_summary": [
    {
      "start_date": "2022-03-31",
      "end_date": "2022-06-29",
      "total_orders": 120378,
      "cancelled_orders": 17185,
      "total_units_sold": 110992,
      "gross_order_value": "78592678.30",
      "net_revenue": "71673394.00",
      "average_order_value": "694.56",
      "cancellation_rate": "14.28"
    }
  ],
  "top_categories": [
    {
      "Category": "Set",
      "total_orders": 47845,
      "units_sold": 43033,
      "net_revenue": "35731673.00",
      "avg_item_value": "834.01",
      "cancelled_orders": 6990,
      "cancellation_rate": "14.61"
    },
    {
      "Category": "kurta",
      "total_orders": 46561,
      "units_sold": 42792,
      "net_revenue": "19425870.00",
      "avg_item_value": "456.84",
      "cancelled_orders": 6818,
      "cancellation_rate": "14.64"
    },
    {
      "Category": "Western Dress",
      "total_orders": 14994,
      "units_sold": 13418,
      "net_revenue": "10209590.00",
      "avg_item_value": "763

In [0]:
prompt = f"""
You are an AI Business Analyst for an e-commerce company.

Your job is to analyze ONLY the Databricks evidence provided below.

DATabricks is the source of truth.
Do NOT invent metrics, facts, causes, or trends.

EVIDENCE:
{evidence_json}

Analyze the business using this structure:

## 1. Executive Summary
Summarize the overall business performance.

## 2. Sales Performance
Identify important observations from the KPI and category data.

## 3. Product Performance
Identify notable SKU-level observations.

## 4. Inventory
Identify important inventory risks.

## 5. Fulfillment
Identify important fulfillment observations.

## 6. Anomalies
Identify significant sales anomalies and their dates.

## 7. Business Areas Requiring Attention
List the areas that deserve investigation.

## 8. Recommended Actions
Give evidence-based actions.
Clearly distinguish recommendations from facts.

Rules:
- Use only the supplied evidence.
- Use exact numbers where available.
- Never fabricate missing information.
- Do not claim that one factor caused another unless the evidence proves it.
- If the evidence is insufficient to determine a cause, explicitly say so.
- Do not describe a recommendation as guaranteed to increase sales.
- Keep the response practical and suitable for a business manager.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

## 1. Executive Summary
Between March 31, 2022, and June 29, 2022, the business generated **71,673,394.00** in net revenue across **110,992** units sold and **120,378** total orders. 

Key summary metrics:
* **Gross Order Value:** 78,592,678.30
* **Net Revenue:** 71,673,394.00
* **Total Orders:** 120,378
* **Cancelled Orders:** 17,185
* **Overall Cancellation Rate:** 14.28%
* **Average Order Value (AOV):** 694.56
* **Total Units Sold:** 110,992

---

## 2. Sales Performance
Observations across product categories:

* **Top Revenue Generator:** The **Set** category generated the highest net revenue at **35,731,673.00** across **47,845** total orders and **43,033** units sold, with an average item value of **834.01** and a cancellation rate of **14.61%** (6,990 cancelled orders).
* **Highest Volume Category:** The **kurta** category recorded **46,561** total orders and **42,792** units sold, generating **19,425,870.00** in net revenue, with an average item value of **456.84** and a cancel